# Manual BBox Annotation with `jupyter-bbox-widget`

Reference: [gereleth/jupyter-bbox-widget](https://github.com/gereleth/jupyter-bbox-widget)

This notebook provides a simple manual bounding-box workflow over a local image folder.


## Install

Run once in the notebook environment if needed:
```bash
pip install jupyter_bbox_widget ipywidgets
```
A Jupyter kernel/server restart may be needed after installation.


In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from challenge.scripts.manual_bbox_annotation import (
    discover_images,
    ensure_record,
    load_annotation_store,
    save_record,
    write_store,
)
from jupyter_bbox_widget import BBoxWidget
import ipywidgets as widgets
from IPython.display import display


In [ ]:
IMAGE_DIR = REPO_ROOT / 'example_images'
ANNOTATION_JSON = REPO_ROOT / 'outputs' / 'manual_bbox_annotations.json'
CLASSES = [
    'local newspaper',
    'bank statement',
    'bills or receipt',
    'business card',
    'condom box',
    'credit or debit card',
    'doctors prescription',
    'letters with address',
    'medical record document',
    'pregnancy test',
    'empty pill bottle',
    'tattoo sleeve',
    'transcript',
    'mortgage or investment report',
    'condom with plastic bag',
    'pregnancy test box',
]
IMAGE_PATTERNS = ('*.jpg', '*.jpeg', '*.png')


In [ ]:
image_paths = discover_images(IMAGE_DIR, IMAGE_PATTERNS)
store = load_annotation_store(ANNOTATION_JSON)
if not image_paths:
    raise FileNotFoundError(f'No images found under: {IMAGE_DIR}')

state = {'index': 0}
status = widgets.HTML()
counter = widgets.HTML()
prev_button = widgets.Button(description='Prev', button_style='')
next_button = widgets.Button(description='Next', button_style='')
save_button = widgets.Button(description='Save', button_style='success')
widget = BBoxWidget(image=str(image_paths[0]), classes=CLASSES)


In [ ]:
def load_current_image():
    image_path = image_paths[state['index']]
    record = ensure_record(store, image_path=image_path, image_root=IMAGE_DIR)
    widget.image = str(image_path)
    widget.classes = CLASSES
    widget.bboxes = list(record.get('bboxes', []))
    counter.value = f'<b>{state["index"] + 1}</b> / {len(image_paths)} : {image_path.name}'
    status.value = f'Loaded: {image_path}'


def save_current(*_):
    image_path = image_paths[state['index']]
    record = save_record(
        store,
        image_path=image_path,
        image_root=IMAGE_DIR,
        classes=CLASSES,
        bboxes=widget.bboxes,
    )
    write_store(store, ANNOTATION_JSON)
    status.value = f'Saved {len(record["bboxes"])} boxes to {ANNOTATION_JSON}'


def go(delta):
    save_current()
    state['index'] = max(0, min(len(image_paths) - 1, state['index'] + delta))
    load_current_image()


prev_button.on_click(lambda _: go(-1))
next_button.on_click(lambda _: go(1))
save_button.on_click(save_current)
widget.on_submit(save_current)
widget.on_skip(lambda: go(1))
load_current_image()
display(widgets.VBox([counter, widgets.HBox([prev_button, save_button, next_button]), status, widget]))


## Notes

- Current annotations are stored in `ANNOTATION_JSON`.
- `Submit` triggers save. `Skip` saves and advances.
- You can edit `CLASSES`, `IMAGE_DIR`, and `ANNOTATION_JSON` to fit your dataset.
- The saved format is a lightweight JSON store intended for manual review workflows.
